In [ ]:
from google.colab import files
files.upload()  # select your kaggle.json when prompted

import os
os.makedirs('/root/.kaggle', exist_ok=True)
!mv kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

Saving kaggle.json to kaggle.json


In [ ]:
DATA_DIR = '/content/drive/MyDrive/cassava_data'
if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR, exist_ok=True)
    !kaggle competitions download -c cassava-leaf-disease-classification -p {DATA_DIR}
    !unzip -q {DATA_DIR}/cassava-leaf-disease-classification.zip -d {DATA_DIR}

100% 5.76G/5.76G [01:08<00:00, 90.8MB/s]



In [ ]:
!kaggle competitions download -c cassava-leaf-disease-classification -p {DATA_DIR}
!unzip -q {DATA_DIR}/cassava-leaf-disease-classification.zip -d {DATA_DIR}

100% 5.76G/5.76G [01:03<00:00, 97.5MB/s]



In [ ]:
!ls {DATA_DIR}


cassava-leaf-disease-classification.zip  test_images	 train_images
label_num_to_disease_map.json		 test_tfrecords  train_tfrecords
sample_submission.csv			 train.csv


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.utils.class_weight import compute_class_weight
from PIL import Image
import torchvision.transforms as T
import timm

df = pd.read_csv(f'{DATA_DIR}/train.csv')
print(df['label'].value_counts())

classes = sorted(df['label'].unique())
class_weights = compute_class_weight('balanced', classes=np.array(classes), y=df['label'].values)
class_weights = torch.tensor(class_weights, dtype=torch.float32)

label
3    13158
4     2577
2     2386
1     2189
0     1087
Name: count, dtype: int64


In [ ]:
train_transform = T.Compose([
    T.RandomResizedCrop(224, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(20),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
val_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class CassavaDataset(Dataset):
    def __init__(self, df, img_dir, transform):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(f"{self.img_dir}/{row['image_id']}").convert('RGB')
        return self.transform(img), row['label']

from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(df, test_size=0.15, stratify=df['label'], random_state=42)

train_ds = CassavaDataset(train_df, f'{DATA_DIR}/train_images', train_transform)
val_ds = CassavaDataset(val_df, f'{DATA_DIR}/train_images', val_transform)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=5).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

CKPT_DIR = '/content/drive/MyDrive/cassava_checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.set_grad_enabled(train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            if train: optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            correct += (out.argmax(1) == labels).sum().item()
            total += imgs.size(0)
    return total_loss/total, correct/total

for epoch in range(10):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    scheduler.step()
    print(f"Epoch {epoch+1}: train_acc={train_acc:.4f} val_acc={val_acc:.4f}")
    torch.save(model.state_dict(), f'{CKPT_DIR}/epoch_{epoch+1}.pt')  # survives disconnects

model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch 1: train_acc=0.6021 val_acc=0.7093
Epoch 2: train_acc=0.7388 val_acc=0.6931
Epoch 3: train_acc=0.7873 val_acc=0.7396
Epoch 4: train_acc=0.8096 val_acc=0.7576
Epoch 5: train_acc=0.8339 val_acc=0.7903
Epoch 6: train_acc=0.8601 val_acc=0.8031
Epoch 7: train_acc=0.8986 val_acc=0.8025
Epoch 8: train_acc=0.9167 val_acc=0.8053
Epoch 9: train_acc=0.9334 val_acc=0.8196
Epoch 10: train_acc=0.9392 val_acc=0.8187


In [ ]:
!ls /content/drive

MyDrive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch, timm
CKPT_DIR = '/content/drive/MyDrive/cassava_checkpoints'
model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=5)
model.load_state_dict(torch.load(f'{CKPT_DIR}/epoch_9.pt', map_location='cpu'))
model.eval()
print("Loaded successfully")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/cassava_checkpoints/epoch_9.pt'

In [ ]:
import os
print(os.path.exists('/content/drive/MyDrive/cassava_checkpoints'))
print(os.listdir('/content/drive/MyDrive/cassava_checkpoints'))

False


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/cassava_checkpoints'

In [ ]:
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os
print(os.listdir('/content/drive/MyDrive'))

['Colab Notebooks', 'Yes (1).gdoc', 'Yes.gdoc', 'MARY NIFEMI OLUROPO.gdoc', 'projeckt', 'CVE_401_Report_Assignment.gdoc', 'XAL_Preisliste_2025_Katalogversion_AT_Rev01 (1).pdf', 'Produkteliste.Erste.Bank.2025 - ASCs final.xlsx', 'XAL_Completed_Project.pdf', 'XAL_Completed_Project.xlsx', 'XAL_Completed_Project.gsheet', 'XAL_Draft_Task600_Portrait.xlsx', 'XAL_Draft_Task600_Portrait.gsheet', 'XAL_Draft_Task600_Portrait_Fixed.xlsx', 'XAL_Draft_Task600_Portrait_Fixed.gsheet', 'XAL_Final_Draft_With_Images.xlsx', 'XAL_Final_Draft_With_Images.gsheet', 'Mary.Oluropo.7th Feb 2026.gdoc', 'Untitled spreadsheet.gsheet', 'Wedding Planner Rosy (1).gsheet', 'Real.gsheet', 'Wedding Planner Rosy.gsheet', 'Gemini Export March 4, 2026 at 2:23:12\u202fPM UTC+1.gdoc', 'Gemini Export March 4, 2026 at 2:23:10\u202fPM UTC+1.gdoc', 'Wedding Planner.gsheet', 'INSTRUCTIONS FOR THE WEDDING PLANNER.gdoc', 'INSTRUCTIONS FOR THE WEDDING PLANNER.pdf', 'Copy of Wedding Planner Rosy.gsheet', 'A Guide to the Wedding Plann

In [ ]:
!cat /content/drive/MyDrive/.file_revisions 2>/dev/null || echo "checking..."

checking...


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/cassava_data', exist_ok=True)
assert 'cassava_data' in os.listdir('/content/drive/MyDrive'), "MOUNT FAILED — STOP, DO NOT PROCEED"
print("Drive confirmed mounted and writable.")

Mounted at /content/drive
Drive confirmed mounted and writable.


In [ ]:
from google.colab import files
files.upload()  # select kaggle.json

os.makedirs('/root/.kaggle', exist_ok=True)
!mv kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

Saving kaggle.json to kaggle.json


In [ ]:
DATA_DIR = '/content/drive/MyDrive/cassava_data'
!kaggle competitions download -c cassava-leaf-disease-classification -p {DATA_DIR}
!unzip -q {DATA_DIR}/cassava-leaf-disease-classification.zip -d {DATA_DIR}
!ls {DATA_DIR}  # confirm train_images, train.csv are there before continuing

cassava-leaf-disease-classification.zip: Skipping, found more recently modified local copy (use --force to force download)
replace /content/drive/MyDrive/cassava_data/label_num_to_disease_map.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: cassava-leaf-disease-classification.zip  test_images	 train_images
label_num_to_disease_map.json		 test_tfrecords
sample_submission.csv			 train.csv


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.utils.class_weight import compute_class_weight
from PIL import Image
import torchvision.transforms as T
import timm

df = pd.read_csv(f'{DATA_DIR}/train.csv')
print(df['label'].value_counts())

classes = sorted(df['label'].unique())
class_weights = compute_class_weight('balanced', classes=np.array(classes), y=df['label'].values)
class_weights = torch.tensor(class_weights, dtype=torch.float32)

label
3    13158
4     2577
2     2386
1     2189
0     1087
Name: count, dtype: int64


In [ ]:
train_transform = T.Compose([
    T.RandomResizedCrop(224, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(20),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
val_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class CassavaDataset(Dataset):
    def __init__(self, df, img_dir, transform):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(f"{self.img_dir}/{row['image_id']}").convert('RGB')
        return self.transform(img), row['label']

from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(df, test_size=0.15, stratify=df['label'], random_state=42)

train_ds = CassavaDataset(train_df, f'{DATA_DIR}/train_images', train_transform)
val_ds = CassavaDataset(val_df, f'{DATA_DIR}/train_images', val_transform)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=5).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

CKPT_DIR = '/content/drive/MyDrive/cassava_checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.set_grad_enabled(train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            if train: optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            correct += (out.argmax(1) == labels).sum().item()
            total += imgs.size(0)
    return total_loss/total, correct/total

for epoch in range(10):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    scheduler.step()
    print(f"Epoch {epoch+1}: train_acc={train_acc:.4f} val_acc={val_acc:.4f}")
    torch.save(model.state_dict(), f'{CKPT_DIR}/epoch_{epoch+1}.pt')  # survives disconnects

model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

FileNotFoundError: Caught FileNotFoundError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/tmp/ipykernel_849/3512493238.py", line 25, in __getitem__
    img = Image.open(f"{self.img_dir}/{row['image_id']}").convert('RGB')
          ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/PIL/Image.py", line 3513, in open
    fp = builtins.open(filename, "rb")
FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/cassava_data/train_images/351189555.jpg'


In [ ]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True Tesla T4


In [ ]:
import os
print(os.path.exists('/content/drive/MyDrive/cassava_data/train_images'))
print(len(os.listdir('/content/drive/MyDrive/cassava_data/train_images')) if os.path.exists('/content/drive/MyDrive/cassava_data/train_images') else "folder missing")

True
8132


In [ ]:
import os
zip_path = '/content/drive/MyDrive/cassava_data/cassava-leaf-disease-classification.zip'
print(os.path.exists(zip_path))
print(os.path.getsize(zip_path) / (1024**3), "GB")  # should be close to 5.76 GB

True
5.760847050696611 GB


In [ ]:
!unzip -q {DATA_DIR}/cassava-leaf-disease-classification.zip -d {DATA_DIR}
!ls {DATA_DIR}/train_images | wc -l

replace /content/drive/MyDrive/cassava_data/label_num_to_disease_map.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: 8132


In [ ]:
import os
os.makedirs('/content/cassava_local', exist_ok=True)
!unzip -q /content/drive/MyDrive/cassava_data/cassava-leaf-disease-classification.zip -d /content/cassava_local
!ls /content/cassava_local/train_images | wc -l

21397


In [ ]:
DATA_DIR_LOCAL = '/content/cassava_local'
df = pd.read_csv(f'{DATA_DIR_LOCAL}/train.csv')

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.utils.class_weight import compute_class_weight
from PIL import Image
import torchvision.transforms as T
import timm

df = pd.read_csv(f'{DATA_DIR}/train.csv')
print(df['label'].value_counts())  # confirm the imbalance before training

classes = sorted(df['label'].unique())
class_weights = compute_class_weight('balanced', classes=np.array(classes), y=df['label'].values)
class_weights = torch.tensor(class_weights, dtype=torch.float32)

label
3    13158
4     2577
2     2386
1     2189
0     1087
Name: count, dtype: int64


In [ ]:
train_transform = T.Compose([
    T.RandomResizedCrop(224, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(20),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
val_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class CassavaDataset(Dataset):
    def __init__(self, df, img_dir, transform):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(f"{self.img_dir}/{row['image_id']}").convert('RGB')
        return self.transform(img), row['label']

from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(df, test_size=0.15, stratify=df['label'], random_state=42)

train_ds = CassavaDataset(train_df, f'{DATA_DIR}/train_images', train_transform)
val_ds = CassavaDataset(val_df, f'{DATA_DIR}/train_images', val_transform)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=5).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

CKPT_DIR = '/content/drive/MyDrive/cassava_checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.set_grad_enabled(train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            if train: optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            correct += (out.argmax(1) == labels).sum().item()
            total += imgs.size(0)
    return total_loss/total, correct/total

for epoch in range(10):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    scheduler.step()
    print(f"Epoch {epoch+1}: train_acc={train_acc:.4f} val_acc={val_acc:.4f}")
    torch.save(model.state_dict(), f'{CKPT_DIR}/epoch_{epoch+1}.pt')  # survives disconnects

FileNotFoundError: Caught FileNotFoundError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/tmp/ipykernel_849/3512493238.py", line 25, in __getitem__
    img = Image.open(f"{self.img_dir}/{row['image_id']}").convert('RGB')
          ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/PIL/Image.py", line 3513, in open
    fp = builtins.open(filename, "rb")
FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/cassava_data/train_images/33138536.jpg'


In [ ]:
import os
print(os.listdir('/content/cassava_local'))
print(os.listdir('/content/cassava_local/train_images')[:5])  # sample of actual filenames

['train_tfrecords', 'train_images', 'label_num_to_disease_map.json', 'train.csv', 'sample_submission.csv', 'test_tfrecords', 'test_images']
['711239502.jpg', '325653540.jpg', '363909221.jpg', '2091719066.jpg', '525742373.jpg']


In [ ]:
train_transform = T.Compose([
    T.RandomResizedCrop(224, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(20),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
val_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

IMG_DIR_LOCAL = '/content/cassava_local/train_images'  # <-- confirm this matches the listing above

class CassavaDataset(Dataset):
    def __init__(self, df, img_dir, transform):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(f"{self.img_dir}/{row['image_id']}").convert('RGB')
        return self.transform(img), row['label']

from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(df, test_size=0.15, stratify=df['label'], random_state=42)

train_ds = CassavaDataset(train_df, IMG_DIR_LOCAL, train_transform)
val_ds = CassavaDataset(val_df, IMG_DIR_LOCAL, val_transform)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)

print(train_ds.img_dir)  # sanity check — must print /content/cassava_local/train_images

/content/cassava_local/train_images


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=5).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

CKPT_DIR = '/content/drive/MyDrive/cassava_checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.set_grad_enabled(train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            if train: optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            correct += (out.argmax(1) == labels).sum().item()
            total += imgs.size(0)
    return total_loss/total, correct/total

for epoch in range(10):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    scheduler.step()
    print(f"Epoch {epoch+1}: train_acc={train_acc:.4f} val_acc={val_acc:.4f}")
    torch.save(model.state_dict(), f'{CKPT_DIR}/epoch_{epoch+1}.pt')

Epoch 1: train_acc=0.5997 val_acc=0.7215
Epoch 2: train_acc=0.7260 val_acc=0.7255
Epoch 3: train_acc=0.7733 val_acc=0.7321
Epoch 4: train_acc=0.8038 val_acc=0.7692
Epoch 5: train_acc=0.8241 val_acc=0.7405
Epoch 6: train_acc=0.8511 val_acc=0.7994
Epoch 7: train_acc=0.8825 val_acc=0.7922
Epoch 8: train_acc=0.9029 val_acc=0.7969
Epoch 9: train_acc=0.9254 val_acc=0.8072
Epoch 10: train_acc=0.9354 val_acc=0.8115


In [ ]:
import os
print(os.listdir('/content/drive/MyDrive/cassava_checkpoints'))

['epoch_1.pt', 'epoch_2.pt', 'epoch_3.pt', 'epoch_4.pt', 'epoch_5.pt', 'epoch_6.pt', 'epoch_7.pt', 'epoch_8.pt', 'epoch_9.pt', 'epoch_10.pt']


In [ ]:
import torch, timm
model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=5)
model.load_state_dict(torch.load('/content/drive/MyDrive/cassava_checkpoints/epoch_10.pt', map_location='cpu'))
model.eval()
print("Loaded successfully — CV pipeline complete.")

Loaded successfully — CV pipeline complete.
